In [1]:
from IPython.display import display, HTML
display(HTML("""
<style>
div.container{width:86% !important;}
div.cell.code_cell.rendered{width:100%;}
div.CodeMirror {font-family:Consolas; font-size:12pt;}
div.output {font-size:15pt; font-weight:bold;}
div.input {font-family:Consolas; font-size:12pt;}
div.prompt {min-width:70px;}
div#toc-wrapper{padding-top:120px;}
div.text_cell_render ul li{font-size:12pt;padding:5px;}
table.dataframe{font-size:15px;}
</style>
"""))

# <font color='red'>ch09_RAG</font>
# 문장 -> 벡터(1차원 숫자 배열) [8.1,8.8,9.1,......2.2]

- openAI API : https://platform.openai.com/ 키를 .env에 등록하여 활용
- upstage : https://console.upstage.ai/ 키를 .env에 등록하여 활용

# 1. 환경변수 load

In [6]:
from dotenv import load_dotenv
import os
load_dotenv()
UPSTAGE_API_KEY = os.getenv("UPSTAGE_API_KEY")
# print(UPSTAGE_API_KEY) # 작동 확인 완료

# 2. 유사도 계산 함수 : https://www.pinecone.io/learn/vector-similarity 
## 1. 유클리드 거리 : 두 벡터 간의 거리 가까운 정도
## 2. cos similarity : 두 벡터 간의 각도(방향) 유사도
## 3. dot product : 두 벡터 간의 곱을 사용하여 거리와 각도(방향)을 모두 고려

In [7]:
import numpy as np

def cosine_similarity(vec1, vec2):
    """두 백터 사이의 코사인 유사도 계산"""
    dot_product = np.dot(vec1, vec2)
    norm_vec1 = np.linalg.norm(vec1) # 벡터의 길이
    norm_vec2 = np.linalg.norm(vec2) 
    if norm_vec1==0 or norm_vec2==0:
        return 0.0
    return dot_product / (norm_vec1*norm_vec2)

# 3. openAI API의 embedding model 사용

In [9]:
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [ ]:
#text-embedding-3-large

In [10]:
openai_response = openai_client.embeddings.create(
    input="King",
    model="text-embedding-3-large"
)

In [12]:
print(openai_response.data[0].embedding)

[0.011214404366910458, 0.013136419467628002, -0.013088766485452652, 0.0033158736769109964, -0.007620553020387888, 0.013009344227612019, 0.02821073867380619, -0.01010249461978674, 0.006087705958634615, 0.036438871175050735, -0.005877237301319838, 0.007112251129001379, -0.04034643992781639, -0.04409516230225563, 0.025796305388212204, 0.004582656547427177, -0.06665104627609253, 0.003496559103950858, -0.02979918196797371, 0.018108244985342026, 0.02725767344236374, -0.05737454444169998, -0.018108244985342026, -0.010984079912304878, 0.021205706521868706, -0.02903672866523266, -0.005619115196168423, 0.009109717793762684, -0.008244016207754612, 0.025907497853040695, -0.006401423364877701, 0.04955940693616867, 0.030974628403782845, 0.05032185837626457, -0.02697175368666649, 0.013469992205500603, 0.024017250165343285, -0.004213343840092421, -0.019236039370298386, -0.023636024445295334, -0.009840400889515877, -0.0283854678273201, -0.07154344767332077, 0.0020322136115282774, 0.017742902040481567, 

In [14]:
import numpy as np
king_vector = np.array(openai_response.data[0].embedding)
print(king_vector.shape)
print(king_vector)

(3072,)
[ 0.0112144   0.01313642 -0.01308877 ...  0.01252487  0.0082599
 -0.00754907]


In [15]:
queen_response = openai_client.embeddings.create(
    input="queen",
    model="text-embedding-3-large"
)

In [16]:
queen_vector = np.array(queen_response.data[0].embedding)
print(queen_vector.shape)
print(queen_vector)

(3072,)
[-0.01385735  0.0008602  -0.0167823  ...  0.00017693  0.01159847
  0.00638929]


In [17]:
king_queen_similarity = cosine_similarity(king_vector, queen_vector)
print("King과 queen의 유사도 :",king_queen_similarity)

0.5260070173168779


In [18]:
slave_response = openai_client.embeddings.create(
    input="slave",
    model="text-embedding-3-large"
)

In [19]:
slave_vector = np.array(slave_response.data[0].embedding)
slave_vector

array([-0.02000881,  0.00614965,  0.01193179, ...,  0.00095144,
       -0.02677098, -0.00585972], shape=(3072,))

In [21]:
king_slave_similarity = cosine_similarity(king_vector, slave_vector)
print("King과 slave의 유사도 :",king_slave_similarity)

0.2470962163268184


In [22]:
print("King과 queen의 유사도 :",king_queen_similarity)
print("King과 slave의 유사도 :",king_slave_similarity)

King과 queen의 유사도 : 0.5260070173168779
King과 slave의 유사도 : 0.2470962163268184


In [ ]:
# 한국어 문장을 벡터로 바꿔도 유사도는 비슷해야 할 듯

In [23]:
kor_king_response = openai_client.embeddings.create(
    input="왕",
    model="text-embedding-3-large"
)

In [24]:
kor_king_vector = np.array(kor_king_response.data[0].embedding)
print(kor_king_vector.shape)
print(kor_king_vector)

(3072,)
[-0.0059576   0.01159042 -0.0131534  ... -0.00355561  0.01320753
 -0.0008204 ]


In [25]:
king_kor_king_similarity = cosine_similarity(king_vector, kor_king_vector)
print(king_kor_king_similarity)

0.5228444035263357


In [26]:
kor_queen_response = openai_client.embeddings.create(
    input="여왕",
    model="text-embedding-3-large"
)
kor_queen_vector = np.array(kor_queen_response.data[0].embedding)
print(kor_queen_vector.shape)
print(kor_queen_vector)

(3072,)
[-0.01307151 -0.00921458 -0.00532257 ... -0.00482468 -0.00204418
  0.02035061]


In [30]:
# 왕과 여왕의 유사도
kor_king_queen_similarity = cosine_similarity(kor_king_vector,kor_queen_vector)
print(kor_king_queen_similarity)

0.48735021148486823


In [28]:
kor_slave_response = openai_client.embeddings.create(
    input="거지",
    model="text-embedding-3-large"
)
kor_slave_vector = np.array(kor_slave_response.data[0].embedding)
print(kor_slave_vector.shape)
print(kor_slave_vector)

(3072,)
[-0.02400834 -0.02815736 -0.00371585 ...  0.01028707 -0.00947125
  0.03754314]


In [29]:
# 왕과 거지의 유사도
kor_king_slave_similarity = cosine_similarity(kor_king_vector,kor_slave_vector)
print(kor_king_slave_similarity)

0.2552309181197065


In [31]:
# king과 왕의 유사도
cosine_similarity(king_vector, kor_king_vector)

np.float64(0.5228444035263357)

# 4. upstage의 embedding model 사용
- 한국 기업 upstage로 한국어 embedding 특화

pip install openai
 
from openai import OpenAI # openai==1.52.2
 
client = OpenAI(
    api_key=UPSTAGE_API_KEY,
    base_url="https://api.upstage.ai/v1"
)
 
response = client.embeddings.create(
    input="Solar embeddings are awesome",
    model="embedding-query"
)
 
print(response.data[0].embedding)

In [32]:
from openai import OpenAI # openai==1.52.2

upstage_client = OpenAI(api_key=UPSTAGE_API_KEY, base_url="https://api.upstage.ai/v1" )

In [33]:
upstage_king_response = upstage_client.embeddings.create( input="king",
                                                         model="embedding-query" )

In [36]:
up_king_vector = np.array(upstage_king_response.data[0].embedding)
print(up_king_vector.shape)
print(up_king_vector)

(4096,)
[-0.01187134 -0.02062988 -0.00674057 ... -0.01081848  0.00247955
  0.01520538]


In [37]:
upstage_queen_response = upstage_client.embeddings.create( input="queen",
                                                         model="embedding-query" )
up_queen_vector = np.array(upstage_queen_response.data[0].embedding)
print(up_queen_vector.shape)

[-0.00164413 -0.00951385 -0.00470734 ...  0.00989532 -0.00730133
  0.02600098]


In [39]:
upstage_kor_king_response = upstage_client.embeddings.create( input="왕",
                                                         model="embedding-query" )
up_kor_king_vector = np.array(upstage_kor_king_response.data[0].embedding)
print(up_kor_king_vector.shape)

(4096,)


In [40]:
upstage_kor_queen_response = upstage_client.embeddings.create( input="여왕",
                                                         model="embedding-query" )
up_kor_queen_vector = np.array(upstage_kor_queen_response.data[0].embedding)
print(up_kor_queen_vector.shape)

(4096,)


In [41]:
cosine_similarity(up_king_vector,up_queen_vector)

np.float64(0.6279103035110143)

In [42]:
cosine_similarity(up_king_vector,up_kor_king_vector) # 같은 뜻을 가진 영어와 한글의 결과

np.float64(0.852188301320326)

In [43]:
cosine_similarity(up_king_vector,up_kor_queen_vector)

np.float64(0.5647304430890209)

In [44]:
cosine_similarity(up_kor_king_vector,up_kor_queen_vector)

np.float64(0.6812030702716614)